In [ ]:
# Google Colab setup: fetch this repository and use this notebook's directory.
from pathlib import Path
import os

REPO_ROOT = Path('/content/BITS_programming')
if not REPO_ROOT.exists():
    !git clone https://github.com/aqwertyuiop48/BITS_programming.git /content/BITS_programming

NOTEBOOK_DIR = REPO_ROOT / 'assignments/assignment_20'
os.chdir(NOTEBOOK_DIR)
print(f'Working directory: {NOTEBOOK_DIR}')

# Unified AWS MLOps Lab v5

Merges Lessons 1–4 into one real AWS workflow: S3 → Managed MLflow → SageMaker Pipeline (Process/Train/Evaluate/Gate/Register) → CodeBuild+ECR → Endpoint with Data Capture → drift on captured requests → CloudWatch alarm → SNS → Lambda → auto-retrain.

## Read this first: SDK v2 is required

This notebook uses `sagemaker.workflow`, `sagemaker.sklearn`, `sagemaker.processing`, and `sagemaker.model`. **SageMaker Python SDK v3 removed every one of those modules** — on v3, all of Block 1 fails immediately with `ModuleNotFoundError`. v3 restructured the package into `core` / `train` / `serve` / `mlops` / `ai_registry`.

**Block 0 (Preflight) checks this for you and tells you exactly what to run.** If you need the fix now:

```
%pip install -q "sagemaker>=2.230,<3"
```
then **restart the kernel**. Verified working on `sagemaker 2.257.5`.

## Bugs fixed across v3 → v5

| # | Bug | Consequence if unfixed |
|---|---|---|
| 1 | Register step used `Model(image_uri=training_image_uri)` with no inference entry point | `model.tar.gz` had no serving code and `SAGEMAKER_PROGRAM` was never set → endpoint deploys but **serving fails**. Now uses `SKLearnModel(entry_point=...)`, which repacks the artifact with `inference.py`. |
| 2 | Drift batch was sampled from the training distribution | KS ≈ 0 → alarm never breaches → **retrain loop never fires**. Block 25b now sends a genuinely shifted batch. |
| 3 | BYOC Flask file was written with broken escaping | Unterminated string literal → **CodeBuild image build fails**. Fixed and the generated file is now syntax-verified. |
| 4 | `sagemaker-mlflow==0.5.0` (unverified pin) | A wrong exact pin **fails the whole training container build**. Now soft-pinned. |
| 5 | No SDK version guard | Silent total failure on SDK v3. Block 0 now catches it. |

## Verification status — what was actually checked

Verified locally against the real SDK (`sagemaker 2.257.5`, `boto3 1.43.50`):

- All 49 notebook cells parse — 0 syntax errors
- All 14 SageMaker imports resolve on v2 (and provably fail on v3)
- All 5 generated scripts written by cells were executed out and syntax-checked — all pass
- The serving handler was **functionally tested**: real RandomForest trained, `model_fn` → `input_fn` → `predict_fn` → `output_fn` returns correct CSV. JSON path works too.
- `image_uris.retrieve("sklearn", "eu-north-1", "1.2-1")` resolves for both training and inference scopes
- `PipelineSession`, `SKLearn`, `SKLearnModel`, `SKLearnProcessor`, `ProcessingStep`, `TrainingStep` all construct and compile

**Not verified:** the `.register()` repack and everything downstream require live S3 credentials, so the pipeline has **not been executed end to end on AWS**. Expect first-pass IAM and quota debugging — normal for this much surface area. Block 0 exists to catch the common causes up front.

**Cost warning:** provisions a Managed MLflow server (~$0.80/hr Small), a real-time `ml.m5.large` endpoint (~$0.10/hr), plus per-job Processing/Training charges and CodeBuild/Lambda/CloudWatch usage. Section 13 tears it all down.

## 0. Preflight — run this FIRST

Fails fast on the things that would otherwise waste 25+ minutes (an MLflow server provisions before you'd discover the problem). Nothing here mutates AWS.

In [1]:
%pip install -q 'sagemaker>=2.230,<3'

Note: you may need to restart the kernel to use updated packages.


In [1]:
# Block 0 - PREFLIGHT. Run before anything else. Read-only.
import sys
import subprocess
from importlib.metadata import version, PackageNotFoundError

problems = []
warnings_ = []

# --- 1. SageMaker SDK major version -------------------------------------------
# This notebook is written against the v2 API (sagemaker.workflow, sagemaker.sklearn,
# sagemaker.processing, sagemaker.model). SDK v3 REMOVED all of those modules --
# every import in Block 1 fails instantly on v3. This is the #1 cause of "nothing works".
try:
    sm_ver = version("sagemaker")
    major = int(sm_ver.split(".")[0])
    if major >= 3:
        problems.append(
            f"sagemaker=={sm_ver} is v3+. This notebook needs v2.\n"
            f"        FIX:  %pip install -q 'sagemaker>=2.230,<3'\n"
            f"        then restart the kernel (Kernel -> Restart Kernel)."
        )
    else:
        print(f"PASS  sagemaker  {sm_ver} (v2 API)")
except PackageNotFoundError:
    problems.append("sagemaker is not installed. FIX: %pip install -q 'sagemaker>=2.230,<3'")

# --- 2. Other required packages -----------------------------------------------
for pkg in ["boto3", "scipy", "pandas", "scikit-learn", "joblib"]:
    try:
        print(f"PASS  {pkg:12s} {version(pkg)}")
    except PackageNotFoundError:
        problems.append(f"{pkg} is not installed. FIX: %pip install -q {pkg}")

# --- 3. Credentials + identity ------------------------------------------------
try:
    import boto3
    from botocore.exceptions import ClientError, NoCredentialsError
    _sts = boto3.client("sts")
    _id = _sts.get_caller_identity()
    print(f"PASS  identity    {_id['Arn']}")
    _account = _id["Account"]
except Exception as e:
    problems.append(f"Cannot resolve AWS identity: {e}")
    _account = None

# --- 4. Region supports Managed MLflow ----------------------------------------
MLFLOW_REGIONS = {
    "us-east-1", "us-east-2", "us-west-2",
    "eu-west-1", "eu-west-2", "eu-west-3", "eu-central-1", "eu-north-1",
    "ap-south-1", "ap-southeast-1", "ap-southeast-2",
    "ap-northeast-1", "ap-northeast-2", "ca-central-1", "sa-east-1",
}
_region = boto3.Session().region_name or "us-east-1"
if _region in MLFLOW_REGIONS:
    print(f"PASS  region      {_region} (Managed MLflow available)")
else:
    warnings_.append(
        f"Region {_region} may not support Managed MLflow. Verify in the console; "
        f"if unsupported, Section 3 will fail."
    )

# --- 5. ml.m5.large quotas ----------------------------------------------------
# These are the exact quotas the Processing / Training / Endpoint steps consume.
# Sandbox accounts commonly have all three at 0, which fails mid-pipeline.
QUOTA_CODES = {
    "L-4EDD3EC6": "ml.m5.large for processing job usage",
    "L-3F19F0A2": "ml.m5.large for training job usage",
    "L-A5A22C0D": "ml.m5.large for endpoint usage",
}
try:
    _sq = boto3.client("service-quotas", region_name=_region)
    for code, label in QUOTA_CODES.items():
        try:
            v = _sq.get_service_quota(ServiceCode="sagemaker", QuotaCode=code)["Quota"]["Value"]
        except ClientError:
            v = _sq.get_aws_default_service_quota(
                ServiceCode="sagemaker", QuotaCode=code
            )["Quota"]["Value"]
        if v < 1:
            problems.append(
                f"Quota '{label}' is {int(v)}. Needs >= 1.\n"
                f"        FIX: Service Quotas -> Amazon SageMaker -> request increase."
            )
        else:
            print(f"PASS  quota       {label} = {int(v)}")
except Exception as e:
    warnings_.append(f"Could not read Service Quotas ({e}). Check manually before running.")

# --- 6. Role trust policy: lambda + codebuild ---------------------------------
# We reuse one execution role for SageMaker, Lambda AND CodeBuild. Default SageMaker
# roles trust only sagemaker.amazonaws.com, so Blocks 21 and 28 fail without this.
try:
    import sagemaker as _sm_sdk
    _role_arn = _sm_sdk.get_execution_role()
    _role_name = _role_arn.split("/")[-1]
    _pol = boto3.client("iam").get_role(RoleName=_role_name)["Role"]["AssumeRolePolicyDocument"]
    _trusted = set()
    for st in _pol.get("Statement", []):
        svc = st.get("Principal", {}).get("Service", [])
        _trusted.update([svc] if isinstance(svc, str) else svc)
    for needed in ["sagemaker.amazonaws.com", "lambda.amazonaws.com", "codebuild.amazonaws.com"]:
        if needed in _trusted:
            print(f"PASS  trust       {needed}")
        else:
            problems.append(
                f"Role {_role_name} does not trust {needed}.\n"
                f"        FIX: IAM -> Roles -> {_role_name} -> Trust relationships -> add it."
            )
except Exception as e:
    warnings_.append(f"Could not verify role trust policy ({e}). Check manually.")

# --- Verdict ------------------------------------------------------------------
print()
if warnings_:
    print("WARNINGS")
    for w in warnings_:
        print("  -", w)
    print()
if problems:
    print("=" * 72)
    print("PREFLIGHT FAILED — fix these before setting ALLOW_AWS_MUTATIONS = True")
    print("=" * 72)
    for p in problems:
        print("  X", p)
    print()
    print("Nothing was created. Re-run this cell after fixing.")
else:
    print("=" * 72)
    print("PREFLIGHT PASSED — safe to proceed.")
    print("=" * 72)

PASS  sagemaker  2.257.5 (v2 API)
PASS  boto3        1.42.97
PASS  scipy        1.16.3
PASS  pandas       2.3.3
PASS  scikit-learn 1.7.2
PASS  joblib       1.5.3
PASS  identity    arn:aws:sts::797715838180:assumed-role/AmazonSageMakerAdminIAMExecutionRole_1/SageMaker
PASS  region      us-east-1 (Managed MLflow available)
sagemaker.config INFO - Fetched defaults config from location: /etc/xdg/sagemaker/config.yaml
sagemaker.config INFO - Not applying SDK defaults from location: /home/sagemaker-user/.config/sagemaker/config.yaml
sagemaker.config INFO - Applied value from config key = SageMaker.PythonSDK.Modules.Session.DefaultS3Bucket
sagemaker.config INFO - Applied value from config key = SageMaker.PythonSDK.Modules.Session.DefaultS3ObjectKeyPrefix
sagemaker.config INFO - Applied value from config key = SageMaker.PythonSDK.Modules.Session.DefaultS3Bucket
sagemaker.config INFO - Applied value from config key = SageMaker.PythonSDK.Modules.Session.DefaultS3ObjectKeyPrefix
PASS  trust      

## 1. Environment Setup

In [2]:
# Block 1 - Imports
import io
import os
import json
import time
import base64
import zipfile
import hashlib
import datetime as dt
from pathlib import Path

import boto3
import joblib
import numpy as np
import pandas as pd
import sagemaker

# --- COMPAT SHIM ---------------------------------------------------------
# sagemaker >= 2.257 dropped `LocalSession` from the top-level `sagemaker`
# namespace, but `sagemaker.workflow.pipeline` still tries to import it as
# `from sagemaker import LocalSession`. Without this shim the next import
# below fails with:
#   ImportError: cannot import name 'LocalSession' from 'sagemaker'
# The shim is a no-op on older sagemaker versions that still export it.
if not hasattr(sagemaker, "LocalSession"):
    try:
        from sagemaker.local import LocalSession as _LocalSession
        sagemaker.LocalSession = _LocalSession
    except Exception:
        # If sagemaker.local is also missing, define a stub so imports proceed.
        # We do not actually use LocalSession anywhere in this notebook.
        class _LocalSession: pass
        sagemaker.LocalSession = _LocalSession
# -------------------------------------------------------------------------

from botocore.exceptions import ClientError
from scipy.stats import ks_2samp
from sklearn.datasets import make_classification
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score

from sagemaker.workflow.pipeline import Pipeline
from sagemaker.workflow.pipeline_context import PipelineSession
from sagemaker.workflow.parameters import ParameterString, ParameterFloat
from sagemaker.workflow.steps import ProcessingStep, TrainingStep
from sagemaker.workflow.condition_step import ConditionStep
from sagemaker.workflow.conditions import ConditionGreaterThanOrEqualTo
from sagemaker.workflow.functions import JsonGet
from sagemaker.workflow.model_step import ModelStep
from sagemaker.workflow.properties import PropertyFile
from sagemaker.sklearn.processing import SKLearnProcessor
from sagemaker.sklearn.estimator import SKLearn
from sagemaker.processing import ProcessingInput, ProcessingOutput
from sagemaker.model import Model

print("Libraries imported")


Libraries imported


In [3]:
# Block 2 - AWS clients + identity
# ------------------------------------------------------------------------
# REGION SELECTOR — change this single line to switch regions everywhere.
# ------------------------------------------------------------------------
# eu-north-1: ml.m5.large processing/training quotas already granted here
#             (from earlier lab work).
# us-east-1:  fresh accounts start with quota = 0 for ml.m5.large in
#             SageMaker Processing/Training/Endpoint. Block 19 will fail
#             with "The account-level service limit ... is 0 Instances"
#             unless you first request a quota increase (2-24h approval).
# ------------------------------------------------------------------------
AWS_REGION_NAME = "eu-north-1"

os.environ["AWS_DEFAULT_REGION"] = AWS_REGION_NAME
os.environ["AWS_REGION"]         = AWS_REGION_NAME

boto_session = boto3.Session(region_name=AWS_REGION_NAME)
region = AWS_REGION_NAME

s3            = boto_session.client("s3",                region_name=region)
sts           = boto_session.client("sts",               region_name=region)
iam           = boto_session.client("iam",               region_name=region)
sm            = boto_session.client("sagemaker",         region_name=region)
runtime       = boto_session.client("sagemaker-runtime", region_name=region)
cloudwatch    = boto_session.client("cloudwatch",        region_name=region)
ecr           = boto_session.client("ecr",               region_name=region)
codebuild     = boto_session.client("codebuild",         region_name=region)
lambda_client = boto_session.client("lambda",            region_name=region)
sns           = boto_session.client("sns",               region_name=region)

identity   = sts.get_caller_identity()
account_id = identity["Account"]

print(f"Region:   {region}")
print(f"Account:  {account_id}")
print(f"Caller:   {identity['Arn']}")


Region:   eu-north-1
Account:  797715838180
Caller:   arn:aws:sts::797715838180:assumed-role/AmazonSageMakerAdminIAMExecutionRole_1/SageMaker


In [4]:
# Block 2b - Check what's already alive in this region
# Read-only. Tells you whether Block 7 will reuse an existing MLflow server
# (fast) or provision a fresh one (25-min wait).
print(f"=== Existing SageMaker resources in {region} ===\n")

_existing_ml = sm.list_mlflow_tracking_servers().get("TrackingServerSummaries", [])
if _existing_ml:
    print("MLflow tracking servers:")
    for _s in _existing_ml:
        _match = " <-- Block 7 will REUSE this" if _s["TrackingServerName"] == mlflow_server_name else ""
        print(f"  {_s['TrackingServerName']}  status={_s['TrackingServerStatus']}  size={_s.get('TrackingServerSize', '?')}{_match}")
else:
    print("MLflow tracking servers: none (Block 7 will provision a fresh one, ~25 min wait)")

_existing_ep = sm.list_endpoints().get("Endpoints", [])
_lab_eps = [e for e in _existing_ep if e["EndpointName"].startswith(PROJECT_NAME)]
if _lab_eps:
    print("\nSageMaker endpoints (matching project):")
    for _e in _lab_eps:
        _match = " <-- Block 24 will UPDATE this" if _e["EndpointName"] == endpoint_name else ""
        print(f"  {_e['EndpointName']}  {_e['EndpointStatus']}{_match}")
else:
    print("\nSageMaker endpoints: none matching this project")

# Cost heads-up if things are still running
_ml_billing = any(s["TrackingServerStatus"] == "Created" for s in _existing_ml)
_ep_billing = any(e["EndpointStatus"] == "InService" for e in _lab_eps)
if _ml_billing or _ep_billing:
    print("\n[BILLING] Some resources above are currently billable:")
    if _ml_billing: print("  - MLflow tracking server: ~$0.80/hr (Small tier)")
    if _ep_billing: print("  - SageMaker endpoint:     ~$0.14/hr (ml.m5.large)")
    print("Section 13 (Block 31) has the guarded teardown.")


=== Existing SageMaker resources in eu-north-1 ===

MLflow tracking servers: none (Block 7 will provision a fresh one, ~25 min wait)

SageMaker endpoints: none matching this project


In [5]:
# Block 3 - Project configuration
PROJECT_NAME = "unified-mlops-mlflow"
ENVIRONMENT  = "dev"
ALLOW_AWS_MUTATIONS = True              # flip to True to actually provision
AUTOMATIC_MODEL_APPROVAL = True
ACCURACY_GATE = 0.80
F1_GATE = 0.35
DRIFT_KS_THRESHOLD = 0.10
DRIFT_FEATURE_COUNT_TO_RETRAIN = 2
ENDPOINT_INSTANCE_TYPE   = "ml.m5.large"
TRAINING_INSTANCE_TYPE   = "ml.m5.large"
PROCESSING_INSTANCE_TYPE = "ml.m5.large"

timestamp = dt.datetime.utcnow().strftime("%Y%m%d-%H%M%S")
bucket_name  = f"{PROJECT_NAME}-{account_id}-{region}".replace("_", "-").lower()
prefix       = f"{PROJECT_NAME}/{ENVIRONMENT}"

pipeline_name             = f"{PROJECT_NAME}-{ENVIRONMENT}-training"
model_package_group_name  = f"{PROJECT_NAME}-{ENVIRONMENT}-models"
mlflow_server_name        = f"{PROJECT_NAME}-{ENVIRONMENT}-mlflow"
mlflow_experiment_name    = f"{PROJECT_NAME}-{ENVIRONMENT}"
endpoint_name             = f"{PROJECT_NAME}-{ENVIRONMENT}-endpoint"
ecr_repo_name             = f"{PROJECT_NAME}-{ENVIRONMENT}-serving"
codebuild_project_name    = f"{PROJECT_NAME}-{ENVIRONMENT}-serving-build"
lambda_function_name      = f"{PROJECT_NAME}-{ENVIRONMENT}-retrain-trigger"
sns_topic_name            = f"{PROJECT_NAME}-{ENVIRONMENT}-drift-topic"
alarm_name                = f"{PROJECT_NAME}-{ENVIRONMENT}-drift-alarm"
namespace                 = f"{PROJECT_NAME}/{ENVIRONMENT}"

local_dir = Path("./local_work")
local_dir.mkdir(exist_ok=True)

print("Bucket:      ", bucket_name)
print("Pipeline:    ", pipeline_name)
print("Endpoint:    ", endpoint_name)
print("MLflow name: ", mlflow_server_name)

Bucket:       unified-mlops-mlflow-797715838180-eu-north-1
Pipeline:     unified-mlops-mlflow-dev-training
Endpoint:     unified-mlops-mlflow-dev-endpoint
MLflow name:  unified-mlops-mlflow-dev-mlflow


In [6]:
# Block 4 - SageMaker execution role
try:
    execution_role = sagemaker.get_execution_role()
except Exception:
    EXECUTION_ROLE_ARN = None    # set manually if not in Studio
    if not EXECUTION_ROLE_ARN:
        raise RuntimeError("Not in SageMaker Studio — set EXECUTION_ROLE_ARN in Block 4.")
    execution_role = EXECUTION_ROLE_ARN
print("Execution role:", execution_role)

sagemaker.config INFO - Applied value from config key = SageMaker.PythonSDK.Modules.Session.DefaultS3Bucket
sagemaker.config INFO - Applied value from config key = SageMaker.PythonSDK.Modules.Session.DefaultS3ObjectKeyPrefix
Execution role: arn:aws:iam::797715838180:role/service-role/AmazonSageMakerAdminIAMExecutionRole_1


## 2. S3 Data Layer

In [7]:
# Block 5 - Create/reuse the S3 bucket
def bucket_exists(name):
    try:
        s3.head_bucket(Bucket=name)
        return True
    except ClientError as e:
        if e.response.get("Error", {}).get("Code") in ("404", "NoSuchBucket"):
            return False
        raise

if ALLOW_AWS_MUTATIONS and not bucket_exists(bucket_name):
    if region == "us-east-1":
        s3.create_bucket(Bucket=bucket_name)
    else:
        s3.create_bucket(
            Bucket=bucket_name,
            CreateBucketConfiguration={"LocationConstraint": region},
        )
    s3.put_bucket_versioning(Bucket=bucket_name,
                             VersioningConfiguration={"Status": "Enabled"})
    print("Bucket created:", bucket_name)
else:
    print("Skipping bucket create (already exists or ALLOW_AWS_MUTATIONS is False).")

Skipping bucket create (already exists or ALLOW_AWS_MUTATIONS is False).


In [8]:
# Block 6 - Loan-risk dataset -> S3
X, y = make_classification(
    n_samples=1500, n_features=6, n_informative=4, n_redundant=1,
    n_classes=2, random_state=42,
)
columns = [
    "income_score", "credit_history_score", "debt_ratio_score",
    "employment_score", "savings_score", "repayment_behavior_score",
]
df = pd.DataFrame(X, columns=columns)
df["loan_default_risk"] = y

raw_key = f"{prefix}/data/raw/loan_data.csv"
raw_path = local_dir / "loan_data.csv"
df.to_csv(raw_path, index=False)

if ALLOW_AWS_MUTATIONS:
    s3.upload_file(str(raw_path), bucket_name, raw_key)
    print(f"Uploaded: s3://{bucket_name}/{raw_key}")

with open(local_dir / "baseline_means.json", "w") as f:
    json.dump(df[columns].mean().to_dict(), f)
df.head()

Uploaded: s3://unified-mlops-mlflow-797715838180-eu-north-1/unified-mlops-mlflow/dev/data/raw/loan_data.csv


,income_score,credit_history_score,debt_ratio_score,employment_score,savings_score,repayment_behavior_score,loan_default_risk
0,1.705099,1.004242,0.588320,-0.410098,-0.282672,-2.320010,1
1,0.918796,-0.262210,1.302603,2.873578,-3.560945,-0.354622,0
2,0.526696,0.524000,-1.071553,0.197359,0.052271,-1.531038,0
3,-0.719885,0.205568,-1.516909,-0.078276,0.812511,0.107889,1
4,1.487276,-1.031392,0.447000,0.456336,-1.478771,-2.083008,1


## 3. Managed MLflow Tracking Server  *(real AWS service — ~25 min to provision)*

In [26]:
# Block 7 - Create/reuse the Managed MLflow tracking server
def get_mlflow_server():
    """Direct lookup by name. describe_ raises ResourceNotFound if it doesn't exist."""
    try:
        return sm.describe_mlflow_tracking_server(TrackingServerName=mlflow_server_name)
    except ClientError as e:
        if e.response.get("Error", {}).get("Code") in ("ResourceNotFound", "ValidationException"):
            return None
        raise

mlflow_server = get_mlflow_server()

if mlflow_server is None and ALLOW_AWS_MUTATIONS:
    sm.create_mlflow_tracking_server(
        TrackingServerName=mlflow_server_name,
        ArtifactStoreUri=f"s3://{bucket_name}/{prefix}/mlflow-artifacts",
        TrackingServerSize="Small",
        RoleArn=execution_role,
        AutomaticModelRegistration=True,
    )
    print("MLflow server creation requested — ~25 min to reach 'Created'.")
    print("Re-run this cell later to pick up the ARN.")
elif mlflow_server is not None:
    print("Status:", mlflow_server["TrackingServerStatus"])
    print("ARN:   ", mlflow_server["TrackingServerArn"])
    print("URL:   ", mlflow_server.get("TrackingServerUrl", "(pending)"))
else:
    print("Skipped (ALLOW_AWS_MUTATIONS is False).")

Status: Created
ARN:    arn:aws:sagemaker:eu-north-1:797715838180:mlflow-tracking-server/unified-mlops-mlflow-dev-mlflow
URL:    https://t-33twhsevvwz6.eu-north-1.experiments.sagemaker.aws


In [27]:
# Block 8 - Resolve the tracking URI (used by the training container)
tracking_server_arn = ""
if mlflow_server is not None and mlflow_server["TrackingServerStatus"] == "Created":
    tracking_server_arn = mlflow_server["TrackingServerArn"]
    print("Tracking URI ready:", tracking_server_arn)
else:
    print("MLflow server not yet 'Created'. Training will still run —")
    print("MLflow logging will be silent until Block 7 shows 'Created' and you re-run 8+.")

Tracking URI ready: arn:aws:sagemaker:eu-north-1:797715838180:mlflow-tracking-server/unified-mlops-mlflow-dev-mlflow


In [11]:
# Poll until the MLflow server is ready
import time, datetime as dt

while True:
    s = get_mlflow_server()
    status = s["TrackingServerStatus"] if s else "NotFound"
    print(dt.datetime.utcnow().strftime("%H:%M:%S"), status)
    if status in ("Created", "CreateFailed", "NotFound"):
        break
    time.sleep(60)

if status == "Created":
    print("\nARN:", s["TrackingServerArn"])
    print("URL:", s.get("TrackingServerUrl", "(pending)"))
    print("\nRun Block 8 next.")
elif status == "CreateFailed":
    print("\nFAILED — check the role's S3 access to the artifact store:")
    print(f"  s3://{bucket_name}/{prefix}/mlflow-artifacts")

14:33:10 Creating
14:34:11 Creating
14:35:11 Creating
14:36:11 Creating
14:37:12 Creating
14:38:12 Creating
14:39:13 Creating
14:40:13 Creating
14:41:14 Creating
14:42:14 Creating
14:43:14 Creating
14:44:15 Creating
14:45:15 Creating
14:46:16 Creating
14:47:16 Creating
14:48:16 Creating
14:49:17 Creating
14:50:17 Creating
14:51:18 Created

ARN: arn:aws:sagemaker:eu-north-1:797715838180:mlflow-tracking-server/unified-mlops-mlflow-dev-mlflow
URL: https://t-33twhsevvwz6.eu-north-1.experiments.sagemaker.aws

Run Block 8 next.


## 3b. UI Checkpoints — Prove MLflow is up and populate it

Everything in this section is **read-mostly** (Block 8c writes 5 synthetic runs; nothing else mutates AWS). The classroom rhythm is:

1. Run **Block 8b** — status card + presigned UI button + inline runs table
2. Click the button → real MLflow UI opens in a new tab (empty at first)
3. Run **Block 8c** (optional, for demos) — logs 5 varied runs so the UI has something interesting to show before the pipeline finishes
4. **Re-run Block 8b any time you want a fresh clickable link** (the presigned URL only lives 5 minutes)

You will come back to Block 8b **four times** in this notebook:

| Checkpoint | When | What the UI shows |
|---|---|---|
| 1 | Right after the poll shows `Created` | Empty experiment — proves the service is up |
| 2 | After Block 8c | 5 demo runs — proves logging works from anywhere |
| 3 | After Block 19 (first pipeline execution) | +1 real training-job run with model artifact |
| 4 | After 2–3 more Block 19 executions | Multiple runs, useful Compare view |


In [28]:
# Block 8b - "Proof MLflow is up" cell for students
# Read-only. Re-run any time to refresh the presigned URL (it expires in 5 min).
from IPython.display import display, Markdown, HTML

# --- 1. Server status ---
srv = sm.describe_mlflow_tracking_server(TrackingServerName=mlflow_server_name)
status = srv["TrackingServerStatus"]
arn    = srv["TrackingServerArn"]

display(Markdown(f"""
### MLflow Tracking Server
- **Name:**   `{mlflow_server_name}`
- **Status:** `{status}` {"✅" if status == "Created" else "⏳ (wait for Created)"}
- **ARN:**    `{arn}`
- **Size:**   `{srv.get("TrackingServerSize", "?")}`
- **Artifact store:** `{srv.get("ArtifactStoreUri", "?")}`
"""))

# --- 2. Clickable UI link (presigned, 5-min click window) ---
if status == "Created":
    url = sm.create_presigned_mlflow_tracking_server_url(
        TrackingServerName=mlflow_server_name,
        ExpiresInSeconds=300,                        # max 300 (5 min click window)
        SessionExpirationDurationInSeconds=43200,    # max 43200 (12h session once opened)
    )["AuthorizedUrl"]
    display(HTML(
        f'<a href="{url}" target="_blank" '
        f'style="display:inline-block;padding:10px 18px;background:#232F3E;'
        f'color:#fff;text-decoration:none;border-radius:4px;font-weight:bold;">'
        f'▶ Open MLflow UI (click within 5 min)</a>'
    ))
else:
    display(Markdown(f"⏳ Server not ready — status `{status}`. Re-run this cell after it becomes `Created`."))

# --- 3. Inline proof (runs table, no click needed) ---
import mlflow
mlflow.set_tracking_uri(arn)
try:
    exps = [e for e in mlflow.search_experiments() if e.name != "Default"]
    display(Markdown(f"**Experiments found (excluding Default):** {len(exps)}"))
    for e in exps[:5]:
        display(Markdown(f"- `{e.name}` (id=`{e.experiment_id}`)"))
        runs = mlflow.search_runs(experiment_ids=[e.experiment_id], max_results=5)
        if not runs.empty:
            cols = [c for c in ["run_id", "status", "start_time",
                                "metrics.val_accuracy", "metrics.val_f1"]
                    if c in runs.columns]
            display(runs[cols])
        else:
            display(Markdown("  _(no runs yet — will populate after Block 8c or Block 19)_"))
except Exception as e:
    display(Markdown(f"⚠️ Could not query MLflow: `{e}`"))



### MLflow Tracking Server
- **Name:**   `unified-mlops-mlflow-dev-mlflow`
- **Status:** `Created` ✅
- **ARN:**    `arn:aws:sagemaker:eu-north-1:797715838180:mlflow-tracking-server/unified-mlops-mlflow-dev-mlflow`
- **Size:**   `Small`
- **Artifact store:** `s3://unified-mlops-mlflow-797715838180-eu-north-1/unified-mlops-mlflow/dev/mlflow-artifacts`


**Experiments found (excluding Default):** 1

- `unified-mlops-mlflow-dev` (id=`1`)

,run_id,status,start_time,metrics.val_accuracy,metrics.val_f1
0,25cc436a8a464dc9bd588f24994f249f,FINISHED,2026-07-20 14:51:29.085000+00:00,0.8306,0.7636
1,1fdc3108226b4f758c60336439d07981,FINISHED,2026-07-20 14:51:27.751000+00:00,0.8871,0.7704
2,7446ebd2c93a483f9b8b0f425879e41d,FINISHED,2026-07-20 14:51:26.550000+00:00,0.8684,0.8412
3,9148ccfde6b04737880b8a7fa69465e4,FINISHED,2026-07-20 14:51:24.406000+00:00,0.8130,0.7868
4,a687992ea96040e489fc5a617d7cbb78,FINISHED,2026-07-20 14:51:22.700000+00:00,0.8567,0.7630


In [29]:
# Block 8c - Seed varied runs so the UI has content for the classroom walkthrough
# OPTIONAL. Skip this if you only want to see runs produced by the real pipeline (Block 19).
# Logs 5 synthetic runs directly from the notebook kernel — no training happens here,
# it just proves that the tracking server accepts logs from anywhere.
import mlflow, random

mlflow.set_tracking_uri(arn)
mlflow.set_experiment(mlflow_experiment_name)

random.seed(42)
for n_est, depth in [(50, 3), (100, 5), (150, 8), (200, 10), (300, 12)]:
    with mlflow.start_run(run_name=f"demo-n{n_est}-d{depth}"):
        mlflow.log_param("n_estimators", n_est)
        mlflow.log_param("max_depth", depth)
        mlflow.log_metric("val_accuracy", round(0.78 + random.random() * 0.12, 4))
        mlflow.log_metric("val_f1",       round(0.76 + random.random() * 0.12, 4))

print("5 demo runs logged. Re-run Block 8b to refresh the UI link,")
print("then click the button and open the experiment to see them.")


🏃 View run demo-n50-d3 at: https://eu-north-1.experiments.sagemaker.aws/#/experiments/1/runs/c055ab69071c4012be7f6a326e34a180
🧪 View experiment at: https://eu-north-1.experiments.sagemaker.aws/#/experiments/1
🏃 View run demo-n100-d5 at: https://eu-north-1.experiments.sagemaker.aws/#/experiments/1/runs/a2e83a8957f449e89a58513f8681d999
🧪 View experiment at: https://eu-north-1.experiments.sagemaker.aws/#/experiments/1
🏃 View run demo-n150-d8 at: https://eu-north-1.experiments.sagemaker.aws/#/experiments/1/runs/d03ed06484974d499530562bb0b4c8f1
🧪 View experiment at: https://eu-north-1.experiments.sagemaker.aws/#/experiments/1
🏃 View run demo-n200-d10 at: https://eu-north-1.experiments.sagemaker.aws/#/experiments/1/runs/dd15f7b324424a479d76a09cb8281cd7
🧪 View experiment at: https://eu-north-1.experiments.sagemaker.aws/#/experiments/1
🏃 View run demo-n300-d12 at: https://eu-north-1.experiments.sagemaker.aws/#/experiments/1/runs/5d76b1d3ddb74d2e82be23abee01ce62
🧪 View experiment at: https://eu

## 4. Pipeline Source Scripts

In [30]:
# Block 9 - preprocess.py
processing_dir = local_dir / "processing_src"
processing_dir.mkdir(exist_ok=True)
(processing_dir / "preprocess.py").write_text('''
import argparse
import json
from pathlib import Path
import pandas as pd
from sklearn.model_selection import train_test_split

if __name__ == "__main__":
    parser = argparse.ArgumentParser()
    parser.add_argument("--test-size", type=float, default=0.25)
    args = parser.parse_args()

    inp = Path("/opt/ml/processing/input")
    out = Path("/opt/ml/processing/output")
    (out / "train").mkdir(parents=True, exist_ok=True)
    (out / "test").mkdir(parents=True, exist_ok=True)

    df = pd.read_csv(inp / "loan_data.csv")
    required = [
        "income_score", "credit_history_score", "debt_ratio_score",
        "employment_score", "savings_score", "repayment_behavior_score",
        "loan_default_risk",
    ]
    missing = [c for c in required if c not in df.columns]
    assert not missing, f"Missing columns: {missing}"
    assert df["loan_default_risk"].isin([0, 1]).all()
    assert df.isna().sum().sum() == 0

    train_df, test_df = train_test_split(
        df, test_size=args.test_size, random_state=42, stratify=df["loan_default_risk"]
    )
    train_df.to_csv(out / "train" / "train.csv", index=False)
    test_df.to_csv(out / "test" / "test.csv", index=False)

    baseline = df[required[:-1]].mean().to_dict()
    with open(out / "train" / "baseline_means.json", "w") as f:
        json.dump(baseline, f)
    print("Wrote", len(train_df), "train and", len(test_df), "test rows")
''')
print("Wrote", processing_dir / "preprocess.py")

Wrote local_work/processing_src/preprocess.py


In [31]:
# Block 10 - Training source_dir: train.py + requirements.txt
training_dir = local_dir / "training_src"
training_dir.mkdir(exist_ok=True)

# NOTE: pins are intentionally soft. A wrong exact pin fails the whole training container
# build with a pip resolver error. Verify against your Studio image and tighten if you want
# reproducibility: pip index versions sagemaker-mlflow
(training_dir / "requirements.txt").write_text(
    "mlflow>=2.13,<3\n"
    "sagemaker-mlflow>=0.1.0\n"
)

(training_dir / "train.py").write_text('''
import argparse
import os
import json
from pathlib import Path

import joblib
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, f1_score
from sklearn.model_selection import train_test_split

def _mlflow_available(uri):
    if not uri:
        return False
    try:
        import mlflow
        import sagemaker_mlflow  # noqa: F401 -- required auth plugin
        mlflow.set_tracking_uri(uri)
        return True
    except Exception as exc:
        print("MLflow not usable in this container:", exc)
        return False

if __name__ == "__main__":
    parser = argparse.ArgumentParser()
    parser.add_argument("--n-estimators", type=int, default=100)
    parser.add_argument("--max-depth", type=int, default=6)
    args = parser.parse_args()

    tracking_uri    = os.environ.get("MLFLOW_TRACKING_URI", "")
    experiment_name = os.environ.get("MLFLOW_EXPERIMENT_NAME", "default")

    train_df = pd.read_csv(Path("/opt/ml/input/data/train") / "train.csv")
    feature_cols = [c for c in train_df.columns if c != "loan_default_risk"]
    X_train, X_val, y_train, y_val = train_test_split(
        train_df[feature_cols], train_df["loan_default_risk"],
        test_size=0.2, random_state=42, stratify=train_df["loan_default_risk"],
    )

    use_mlflow = _mlflow_available(tracking_uri)
    if use_mlflow:
        import mlflow, mlflow.sklearn
        mlflow.set_experiment(experiment_name)
        mlflow.start_run(run_name="training-job")
        mlflow.log_param("n_estimators", args.n_estimators)
        mlflow.log_param("max_depth", args.max_depth)

    model = RandomForestClassifier(
        n_estimators=args.n_estimators, max_depth=args.max_depth,
        random_state=42, class_weight="balanced",
    )
    model.fit(X_train, y_train)
    val_pred = model.predict(X_val)
    val_acc = float(accuracy_score(y_val, val_pred))
    val_f1  = float(f1_score(y_val, val_pred))

    if use_mlflow:
        import mlflow, mlflow.sklearn
        from mlflow.models.signature import infer_signature
        mlflow.log_metric("val_accuracy", val_acc)
        mlflow.log_metric("val_f1", val_f1)
        mlflow.sklearn.log_model(
            model, artifact_path="model",
            signature=infer_signature(X_val, val_pred),
            input_example=X_val.head(3),
        )
        mlflow.end_run()
        print("MLflow run logged to", tracking_uri)

    model_dir = Path("/opt/ml/model")
    joblib.dump(model, model_dir / "model.joblib")
    with open(model_dir / "metrics.json", "w") as f:
        json.dump({"val_accuracy": val_acc, "val_f1": val_f1}, f)
    print("Training complete. val_accuracy =", val_acc, "val_f1 =", val_f1)
''')
print("Wrote", training_dir / "train.py")

Wrote local_work/training_src/train.py


In [32]:
# Block 11 - evaluate.py
eval_dir = local_dir / "eval_src"
eval_dir.mkdir(exist_ok=True)
(eval_dir / "evaluate.py").write_text('''
import json
import tarfile
from pathlib import Path
import joblib
import pandas as pd
from sklearn.metrics import accuracy_score, f1_score

if __name__ == "__main__":
    model_dir = Path("/opt/ml/processing/model")
    test_dir  = Path("/opt/ml/processing/test")
    out_dir   = Path("/opt/ml/processing/evaluation")
    out_dir.mkdir(parents=True, exist_ok=True)

    with tarfile.open(model_dir / "model.tar.gz") as tar:
        tar.extractall(model_dir)

    model = joblib.load(model_dir / "model.joblib")
    test_df = pd.read_csv(test_dir / "test.csv")
    feature_cols = [c for c in test_df.columns if c != "loan_default_risk"]
    preds = model.predict(test_df[feature_cols])
    acc = float(accuracy_score(test_df["loan_default_risk"], preds))
    f1  = float(f1_score(test_df["loan_default_risk"], preds))
    with open(out_dir / "evaluation.json", "w") as f:
        json.dump({"metrics": {"accuracy": {"value": acc}, "f1": {"value": f1}}}, f)
    print("Evaluation:", acc, f1)
''')
print("Wrote", eval_dir / "evaluate.py")

Wrote local_work/eval_src/evaluate.py


In [33]:
# Block 11b - inference.py for the SERVING container
# This gets bundled into model.tar.gz by the SKLearnModel repack in Block 16.
# Without it the endpoint has no serving code and CreateEndpoint fails.
inference_dir = local_dir / "inference_src"
inference_dir.mkdir(exist_ok=True)
(inference_dir / "inference.py").write_text('''
import io
import json
from pathlib import Path
import joblib
import pandas as pd

FEATURE_COLS = [
    "income_score", "credit_history_score", "debt_ratio_score",
    "employment_score", "savings_score", "repayment_behavior_score",
]

def model_fn(model_dir):
    return joblib.load(Path(model_dir) / "model.joblib")

def input_fn(request_body, content_type="text/csv"):
    if "json" in content_type:
        payload = json.loads(request_body)
        return pd.DataFrame(payload.get("instances", payload), columns=FEATURE_COLS)
    return pd.read_csv(io.StringIO(request_body), header=None, names=FEATURE_COLS)

def predict_fn(input_data, model):
    return model.predict(input_data)

def output_fn(prediction, accept="text/csv"):
    return "\\n".join(str(int(p)) for p in prediction) + "\\n", "text/csv"
''')
print("Wrote", inference_dir / "inference.py")

Wrote local_work/inference_src/inference.py


## 5. Assemble the SageMaker Pipeline

In [34]:
# Block 12 - Pipeline session and parameters
# after
pipeline_session = PipelineSession(
    default_bucket=bucket_name,
    default_bucket_prefix="",
)

param_input_data      = ParameterString(name="InputDataUri",
                                        default_value=f"s3://{bucket_name}/{raw_key}")
param_accuracy_gate   = ParameterFloat(name="AccuracyGate", default_value=ACCURACY_GATE)
param_f1_gate         = ParameterFloat(name="F1Gate",       default_value=F1_GATE)
param_approval_status = ParameterString(
    name="ModelApprovalStatus",
    default_value="Approved" if AUTOMATIC_MODEL_APPROVAL else "PendingManualApproval",
)
print("Pipeline parameters defined")

Pipeline parameters defined


In [35]:
# Block 13 - Processing step
sklearn_processor = SKLearnProcessor(
    framework_version="1.2-1",
    instance_type=PROCESSING_INSTANCE_TYPE,
    instance_count=1,
    base_job_name=f"{PROJECT_NAME}-preprocess",
    role=execution_role,
    sagemaker_session=pipeline_session,
)
processing_step_args = sklearn_processor.run(
    code=str(processing_dir / "preprocess.py"),
    inputs=[ProcessingInput(source=param_input_data, destination="/opt/ml/processing/input")],
    outputs=[
        ProcessingOutput(output_name="train", source="/opt/ml/processing/output/train"),
        ProcessingOutput(output_name="test",  source="/opt/ml/processing/output/test"),
    ],
)
step_process = ProcessingStep(name="ValidateAndSplit", step_args=processing_step_args)
print("Processing step defined")

Processing step defined


In [36]:
# Block 14 - Training step (source_dir triggers pip install of mlflow + sagemaker-mlflow)
sklearn_estimator = SKLearn(
    source_dir=str(training_dir),
    entry_point="train.py",
    framework_version="1.2-1",
    instance_type=TRAINING_INSTANCE_TYPE,
    role=execution_role,
    base_job_name=f"{PROJECT_NAME}-train",
    sagemaker_session=pipeline_session,
    hyperparameters={"n-estimators": 100, "max-depth": 6},
    environment={
        "MLFLOW_TRACKING_URI":    tracking_server_arn,
        "MLFLOW_EXPERIMENT_NAME": mlflow_experiment_name,
    },
)
training_step_args = sklearn_estimator.fit(inputs={
    "train": step_process.properties.ProcessingOutputConfig.Outputs["train"].S3Output.S3Uri
})
step_train = TrainingStep(name="TrainWithMLflow", step_args=training_step_args)
print("Training step defined")

Training step defined


In [37]:
# Block 15 - Evaluation step
evaluation_report = PropertyFile(
    name="EvaluationReport", output_name="evaluation", path="evaluation.json"
)
eval_processor = SKLearnProcessor(
    framework_version="1.2-1",
    instance_type=PROCESSING_INSTANCE_TYPE,
    instance_count=1,
    base_job_name=f"{PROJECT_NAME}-evaluate",
    role=execution_role,
    sagemaker_session=pipeline_session,
)
eval_step_args = eval_processor.run(
    code=str(eval_dir / "evaluate.py"),
    inputs=[
        ProcessingInput(
            source=step_train.properties.ModelArtifacts.S3ModelArtifacts,
            destination="/opt/ml/processing/model",
        ),
        ProcessingInput(
            source=step_process.properties.ProcessingOutputConfig.Outputs["test"].S3Output.S3Uri,
            destination="/opt/ml/processing/test",
        ),
    ],
    outputs=[ProcessingOutput(output_name="evaluation", source="/opt/ml/processing/evaluation")],
)
step_evaluate = ProcessingStep(
    name="EvaluateCandidateModel",
    step_args=eval_step_args,
    property_files=[evaluation_report],
)
print("Evaluation step defined")

Evaluation step defined


In [38]:
# Block 16 - Register step
# FIX vs v3: v3 used Model(image_uri=training_image_uri, model_data=...) with no inference
# entry point. The model.tar.gz from training holds only model.joblib + metrics.json — no
# serving code — so the endpoint had nothing to load. SKLearnModel + entry_point triggers a
# repack step that bundles inference.py into the artifact and resolves the INFERENCE image.
from sagemaker.sklearn.model import SKLearnModel

model = SKLearnModel(
    model_data=step_train.properties.ModelArtifacts.S3ModelArtifacts,
    role=execution_role,
    entry_point="inference.py",
    source_dir=str(inference_dir),
    framework_version="1.2-1",
    py_version="py3",
    sagemaker_session=pipeline_session,
)
register_args = model.register(
    content_types=["text/csv"],
    response_types=["text/csv"],
    inference_instances=[ENDPOINT_INSTANCE_TYPE],
    transform_instances=[ENDPOINT_INSTANCE_TYPE],
    model_package_group_name=model_package_group_name,
    approval_status=param_approval_status,
)
step_register = ModelStep(name="RegisterModel", step_args=register_args)
print("Register step defined")

Register step defined


In [39]:
# Block 17 - Quality gate
cond_acc = ConditionGreaterThanOrEqualTo(
    left=JsonGet(step_name=step_evaluate.name, property_file=evaluation_report,
                 json_path="metrics.accuracy.value"),
    right=param_accuracy_gate,
)
cond_f1 = ConditionGreaterThanOrEqualTo(
    left=JsonGet(step_name=step_evaluate.name, property_file=evaluation_report,
                 json_path="metrics.f1.value"),
    right=param_f1_gate,
)
step_quality_gate = ConditionStep(
    name="ModelQualityGate",
    conditions=[cond_acc, cond_f1],
    if_steps=[step_register],
    else_steps=[],
)
print("Condition step defined")

Condition step defined


In [40]:
# Block 18 - Upsert the pipeline (WARNS if MLflow URI is empty)
pipeline = Pipeline(
    name=pipeline_name,
    parameters=[param_input_data, param_accuracy_gate, param_f1_gate, param_approval_status],
    steps=[step_process, step_train, step_evaluate, step_quality_gate],
    sagemaker_session=pipeline_session,
)

if not tracking_server_arn:
    print("!! MLFLOW_TRACKING_URI is empty — training runs will not log to MLflow.")
    print("!! To enable MLflow logging: wait for Block 7 to show 'Created',")
    print("!! re-run Block 8, then re-run Blocks 14 and 18 to bake the URI into the pipeline.")

if ALLOW_AWS_MUTATIONS:
    pipeline.upsert(role_arn=execution_role)
    print("Pipeline upserted:", pipeline_name)
else:
    _ = json.loads(pipeline.definition())
    print("Definition compiles OK. Not upserted (ALLOW_AWS_MUTATIONS is False).")

Pipeline upserted: unified-mlops-mlflow-dev-training


## 6. Run the Pipeline

In [41]:
# Block 19 - Start execution and poll to completion
if ALLOW_AWS_MUTATIONS:
    execution = pipeline.start(parameters={
        "InputDataUri": f"s3://{bucket_name}/{raw_key}",
        "AccuracyGate": ACCURACY_GATE,
        "F1Gate":       F1_GATE,
        "ModelApprovalStatus": "Approved" if AUTOMATIC_MODEL_APPROVAL else "PendingManualApproval",
    })
    print("Execution ARN:", execution.arn)
    print("Waiting for pipeline to finish (~10-15 min)...")
    print("Steps: Preprocess -> Train -> Evaluate -> Gate -> Register")
    while True:
        status = execution.describe()["PipelineExecutionStatus"]
        print(dt.datetime.utcnow().isoformat(), status)
        if status in ("Succeeded", "Failed", "Stopped"):
            break
        time.sleep(30)
    for step in execution.list_steps():
        print(step["StepName"], "->", step["StepStatus"])

    # UI CHECKPOINT: the Training step logged a new MLflow run.
    if status == "Succeeded":
        print("\n>>> Re-run Block 8b to refresh the UI link and see the new run.")
        print(">>> In the MLflow UI, open the 'training-job' run to see the logged model artifact.")
    else:
        print("\n>>> Pipeline did not succeed. Run Block 19b for step-level diagnostics.")
else:
    print("Skipped (ALLOW_AWS_MUTATIONS is False).")


Execution ARN: arn:aws:sagemaker:eu-north-1:797715838180:pipeline/unified-mlops-mlflow-dev-training/execution/241xxsovxbiq
Waiting for pipeline to finish (~10-15 min)...
Steps: Preprocess -> Train -> Evaluate -> Gate -> Register
2026-07-20T14:56:42.523098 Executing
2026-07-20T14:57:12.760601 Executing
2026-07-20T14:57:42.959420 Executing
2026-07-20T14:58:13.151371 Executing
2026-07-20T14:58:43.354446 Executing
2026-07-20T14:59:13.546589 Executing
2026-07-20T14:59:43.742570 Executing
2026-07-20T15:00:13.941352 Executing
2026-07-20T15:00:44.136292 Executing
2026-07-20T15:01:14.570583 Executing
2026-07-20T15:01:44.767706 Executing
2026-07-20T15:02:14.965367 Executing
2026-07-20T15:02:45.175565 Executing
2026-07-20T15:03:15.364246 Executing
2026-07-20T15:03:45.568056 Executing
2026-07-20T15:04:15.769657 Executing
2026-07-20T15:04:45.974319 Executing
2026-07-20T15:05:16.179664 Executing
2026-07-20T15:05:46.646634 Succeeded
RegisterModel-RegisterModel -> Succeeded
ModelQualityGate -> Succeed

In [42]:
# Block 19b - Diagnose the failed pipeline execution
exec_arn = execution.arn
steps = sm.list_pipeline_execution_steps(PipelineExecutionArn=exec_arn)["PipelineExecutionSteps"]
for st in steps:
    print(f"\n=== {st['StepName']} -> {st['StepStatus']} ===")
    if st.get("FailureReason"):
        print("FailureReason:", st["FailureReason"])
    md = st.get("Metadata", {})
    tj_arn = md.get("TrainingJob", {}).get("Arn")
    if tj_arn:
        tj_name = tj_arn.split("/")[-1]
        desc = sm.describe_training_job(TrainingJobName=tj_name)
        print("TrainingJobName :", tj_name)
        print("SecondaryStatus :", desc.get("SecondaryStatus"))
        print("FailureReason   :", desc.get("FailureReason"))
        print("CloudWatch logs : /aws/sagemaker/TrainingJobs -> log stream", tj_name, "/algo-1-...")


=== RegisterModel-RegisterModel -> Succeeded ===

=== ModelQualityGate -> Succeeded ===

=== EvaluateCandidateModel -> Succeeded ===

=== TrainWithMLflow -> Succeeded ===
TrainingJobName : pipelines-241xxsovxbiq-TrainWithMLflow-ueclUFniVJ
SecondaryStatus : Completed
FailureReason   : None
CloudWatch logs : /aws/sagemaker/TrainingJobs -> log stream pipelines-241xxsovxbiq-TrainWithMLflow-ueclUFniVJ /algo-1-...

=== ValidateAndSplit -> Succeeded ===


## 7. Containerized Serving Image — CodeBuild → ECR  *(Lesson 4 deliverable)*

**Read this before you run Section 7.** The image below is real: a real Flask app, really built by CodeBuild, really pushed to ECR. But **the endpoint in Section 8 does not use it** — Section 8 deploys from the Model Registry package, which resolves to the AWS-managed SKLearn inference container.

That is deliberate, and it is the standard split:

- **Section 7 (this section)** is the Lesson 4 containerization deliverable — a BYOC image you can inspect, pull, and run locally.
- **Section 8** is the managed-container serving path, which is what Data Capture, the Model Registry, and the retrain loop are wired to.

If you want the endpoint to actually serve *from* this ECR image instead, set `USE_BYOC_ENDPOINT = True` in Block 21b and it will deploy the BYOC container as the serving path. Leave it `False` to keep the managed SKLearn path.

Nothing here is a stub — but don't let a slide claim the endpoint is running this image unless you flip that switch.

In [43]:
# Block 20 - Real Flask inference + Dockerfile + buildspec
docker_dir = local_dir / "docker"
docker_dir.mkdir(exist_ok=True)

(docker_dir / "requirements.txt").write_text(
    "flask==3.0.3\n"
    "gunicorn==22.0.0\n"
    "scikit-learn==1.2.2\n"
    "joblib==1.4.2\n"
    "pandas==2.2.2\n"
    "numpy==1.26.4\n"
)

(docker_dir / "inference.py").write_text('''
import io
import os
import json
from pathlib import Path
import joblib
import pandas as pd
from flask import Flask, request, Response

MODEL_DIR = Path(os.environ.get("MODEL_DIR", "/opt/ml/model"))
_model_path = MODEL_DIR / "model.joblib"
_model = joblib.load(_model_path) if _model_path.exists() else None

FEATURE_COLS = [
    "income_score", "credit_history_score", "debt_ratio_score",
    "employment_score", "savings_score", "repayment_behavior_score",
]

app = Flask(__name__)

@app.route("/ping", methods=["GET"])
def ping():
    return Response(status=200 if _model is not None else 500)

@app.route("/invocations", methods=["POST"])
def invoke():
    content_type = request.content_type or "text/csv"
    raw = request.data.decode("utf-8")
    if "json" in content_type:
        payload = json.loads(raw)
        df = pd.DataFrame(payload.get("instances", payload), columns=FEATURE_COLS)
    else:
        df = pd.read_csv(io.StringIO(raw), header=None, names=FEATURE_COLS)
    preds = _model.predict(df)
    return Response("\\n".join(str(int(p)) for p in preds) + "\\n", mimetype="text/csv")
''')

(docker_dir / "Dockerfile").write_text('''
FROM public.ecr.aws/docker/library/python:3.10-slim
WORKDIR /opt/program
COPY requirements.txt .
RUN pip install --no-cache-dir -r requirements.txt
COPY inference.py .
ENV MODEL_DIR=/opt/ml/model
ENV PYTHONUNBUFFERED=1
EXPOSE 8080
CMD ["gunicorn", "--bind", "0.0.0.0:8080", "--workers", "2", "--timeout", "60", "inference:app"]
''')

(docker_dir / "buildspec.yml").write_text(f'''
version: 0.2
phases:
  pre_build:
    commands:
      - aws ecr get-login-password --region {region} | docker login --username AWS --password-stdin {account_id}.dkr.ecr.{region}.amazonaws.com
      - aws ecr describe-repositories --repository-names {ecr_repo_name} --region {region} || aws ecr create-repository --repository-name {ecr_repo_name} --region {region}
  build:
    commands:
      - docker build -t {ecr_repo_name}:latest .
      - docker tag {ecr_repo_name}:latest {account_id}.dkr.ecr.{region}.amazonaws.com/{ecr_repo_name}:latest
  post_build:
    commands:
      - docker push {account_id}.dkr.ecr.{region}.amazonaws.com/{ecr_repo_name}:latest
''')

zip_path = local_dir / "docker_source.zip"
with zipfile.ZipFile(zip_path, "w") as zf:
    for f in docker_dir.iterdir():
        zf.write(f, arcname=f.name)
print("Docker source zipped:", zip_path)

Docker source zipped: local_work/docker_source.zip


In [44]:
# Block 21 - Create ECR repo + upload source + run CodeBuild
source_key = f"{prefix}/codebuild/docker_source.zip"

if ALLOW_AWS_MUTATIONS:
    try:
        ecr.create_repository(repositoryName=ecr_repo_name,
                              imageScanningConfiguration={"scanOnPush": True})
        print("ECR repo created:", ecr_repo_name)
    except ecr.exceptions.RepositoryAlreadyExistsException:
        print("ECR repo already exists:", ecr_repo_name)

    s3.upload_file(str(zip_path), bucket_name, source_key)

    try:
        codebuild.create_project(
            name=codebuild_project_name,
            source={"type": "S3", "location": f"{bucket_name}/{source_key}"},
            artifacts={"type": "NO_ARTIFACTS"},
            environment={
                "type": "LINUX_CONTAINER",
                "image": "aws/codebuild/standard:7.0",
                "computeType": "BUILD_GENERAL1_SMALL",
                "privilegedMode": True,
            },
            serviceRole=execution_role,
        )
        print("CodeBuild project created")
    except codebuild.exceptions.ResourceAlreadyExistsException:
        codebuild.update_project(
            name=codebuild_project_name,
            source={"type": "S3", "location": f"{bucket_name}/{source_key}"},
        )
        print("CodeBuild project source updated")

    build_id = codebuild.start_build(projectName=codebuild_project_name)["build"]["id"]
    print("Build started:", build_id)
    while True:
        status = codebuild.batch_get_builds(ids=[build_id])["builds"][0]["buildStatus"]
        print(dt.datetime.utcnow().isoformat(), "build:", status)
        if status != "IN_PROGRESS":
            break
        time.sleep(15)
    print("Serving image:", f"{account_id}.dkr.ecr.{region}.amazonaws.com/{ecr_repo_name}:latest")
else:
    print("Skipped.")

ECR repo already exists: unified-mlops-mlflow-dev-serving
CodeBuild project source updated
Build started: unified-mlops-mlflow-dev-serving-build:6105d87c-325c-400b-9a62-1e471b88bc7e
2026-07-20T15:05:51.889537 build: IN_PROGRESS
2026-07-20T15:06:07.031056 build: IN_PROGRESS
2026-07-20T15:06:22.175019 build: IN_PROGRESS
2026-07-20T15:06:37.315332 build: IN_PROGRESS
2026-07-20T15:06:52.459872 build: IN_PROGRESS
2026-07-20T15:07:07.608344 build: SUCCEEDED
Serving image: 797715838180.dkr.ecr.eu-north-1.amazonaws.com/unified-mlops-mlflow-dev-serving:latest


In [45]:
# Block 21b - Choose the serving path for Section 8
# False -> endpoint serves from the Model Registry package (AWS-managed SKLearn container).
# True  -> endpoint serves from the BYOC Flask image built in Blocks 20-21.
USE_BYOC_ENDPOINT = False

byoc_image_uri = f"{account_id}.dkr.ecr.{region}.amazonaws.com/{ecr_repo_name}:latest"
print("Serving path:", "BYOC ECR image" if USE_BYOC_ENDPOINT else "Model Registry package")
print("BYOC image URI:", byoc_image_uri)

Serving path: Model Registry package
BYOC image URI: 797715838180.dkr.ecr.eu-north-1.amazonaws.com/unified-mlops-mlflow-dev-serving:latest


## 8. Deploy the Endpoint with Data Capture

In [46]:
# Block 22 - Latest registered model package
pkgs = []
if ALLOW_AWS_MUTATIONS:
    try:
        pkgs = sm.list_model_packages(
            ModelPackageGroupName=model_package_group_name,
            SortBy="CreationTime", SortOrder="Descending", MaxResults=5,
        ).get("ModelPackageSummaryList", [])
    except ClientError:
        pkgs = []
for p in pkgs:
    print(p["ModelPackageArn"], "-", p["ModelApprovalStatus"])
latest_package_arn = pkgs[0]["ModelPackageArn"] if pkgs else None
print("\nLatest:", latest_package_arn)

arn:aws:sagemaker:eu-north-1:797715838180:model-package/unified-mlops-mlflow-dev-models/10 - Approved
arn:aws:sagemaker:eu-north-1:797715838180:model-package/unified-mlops-mlflow-dev-models/9 - Approved
arn:aws:sagemaker:eu-north-1:797715838180:model-package/unified-mlops-mlflow-dev-models/8 - Approved
arn:aws:sagemaker:eu-north-1:797715838180:model-package/unified-mlops-mlflow-dev-models/7 - Approved
arn:aws:sagemaker:eu-north-1:797715838180:model-package/unified-mlops-mlflow-dev-models/6 - Approved

Latest: arn:aws:sagemaker:eu-north-1:797715838180:model-package/unified-mlops-mlflow-dev-models/10


In [47]:
# Block 23 - Manual approval path (skipped if AUTOMATIC_MODEL_APPROVAL)
if ALLOW_AWS_MUTATIONS and latest_package_arn and not AUTOMATIC_MODEL_APPROVAL:
    sm.update_model_package(ModelPackageArn=latest_package_arn, ModelApprovalStatus="Approved")
    print("Approved:", latest_package_arn)
else:
    print("Skipped.")

Skipped.


In [48]:
# Block 24 - Create endpoint with Data Capture ON (idempotent)
data_capture_s3_uri = f"s3://{bucket_name}/{prefix}/data-capture"
model_name = f"{PROJECT_NAME}-{ENVIRONMENT}-model-{timestamp}"
endpoint_config_name = f"{endpoint_name}-config-{timestamp}"

if ALLOW_AWS_MUTATIONS and latest_package_arn:
    try:
        if USE_BYOC_ENDPOINT:
            # Serve from the Flask image built in Section 7. ModelDataUrl is untarred by
            # SageMaker into /opt/ml/model, which is where inference.py loads model.joblib.
            model_artifact = sm.describe_model_package(
                ModelPackageName=latest_package_arn
            )["InferenceSpecification"]["Containers"][0]["ModelDataUrl"]
            sm.create_model(
                ModelName=model_name,
                PrimaryContainer={
                    "Image": byoc_image_uri,
                    "ModelDataUrl": model_artifact,
                },
                ExecutionRoleArn=execution_role,
            )
            print("Model created from BYOC image:", byoc_image_uri)
        else:
            sm.create_model(
                ModelName=model_name,
                Containers=[{"ModelPackageName": latest_package_arn}],
                ExecutionRoleArn=execution_role,
            )
            print("Model created from Model Registry package")
    except ClientError as e:
        if "already exists" not in str(e):
            raise

    try:
        sm.create_endpoint_config(
            EndpointConfigName=endpoint_config_name,
            ProductionVariants=[{
                "VariantName": "AllTraffic",
                "ModelName": model_name,
                "InstanceType": ENDPOINT_INSTANCE_TYPE,
                "InitialInstanceCount": 1,
            }],
            DataCaptureConfig={
                "EnableCapture": True,
                "InitialSamplingPercentage": 100,
                "DestinationS3Uri": data_capture_s3_uri,
                "CaptureOptions": [
                    {"CaptureMode": "Input"},
                    {"CaptureMode": "Output"},
                ],
                "CaptureContentTypeHeader": {
                    "CsvContentTypes":  ["text/csv"],
                    "JsonContentTypes": ["application/json"],
                },
            },
        )
    except ClientError as e:
        if "already exists" not in str(e):
            raise

    try:
        sm.create_endpoint(EndpointName=endpoint_name, EndpointConfigName=endpoint_config_name)
        print("Endpoint create started:", endpoint_name)
    except ClientError as e:
        if "already exists" in str(e):
            sm.update_endpoint(EndpointName=endpoint_name, EndpointConfigName=endpoint_config_name)
            print("Endpoint update (blue/green) started:", endpoint_name)
        else:
            raise

    while True:
        st = sm.describe_endpoint(EndpointName=endpoint_name)["EndpointStatus"]
        print(dt.datetime.utcnow().isoformat(), "endpoint:", st)
        if st in ("InService", "Failed"):
            break
        time.sleep(30)
    print("Data capture destination:", data_capture_s3_uri)
else:
    print("Skipped.")

Model created from Model Registry package
Endpoint create started: unified-mlops-mlflow-dev-endpoint
2026-07-20T15:07:10.106786 endpoint: Creating
2026-07-20T15:07:40.299844 endpoint: Creating
2026-07-20T15:08:10.477840 endpoint: Creating
2026-07-20T15:08:40.660745 endpoint: Creating
2026-07-20T15:09:10.841718 endpoint: Creating
2026-07-20T15:09:41.007209 endpoint: Creating
2026-07-20T15:10:11.179441 endpoint: Creating
2026-07-20T15:10:41.393206 endpoint: InService
Data capture destination: s3://unified-mlops-mlflow-797715838180-eu-north-1/unified-mlops-mlflow/dev/data-capture


## 9. Invoke the Endpoint  *(populates capture bucket)*

In [49]:
# Block 25 - Invoke the endpoint 50 times to build up captured requests
if ALLOW_AWS_MUTATIONS:
    n_invocations = 50
    sample = df[columns].sample(n_invocations, random_state=1, replace=True)
    for i, (_, row) in enumerate(sample.iterrows()):
        payload = ",".join(str(v) for v in row.values)
        resp = runtime.invoke_endpoint(
            EndpointName=endpoint_name, ContentType="text/csv", Body=payload
        )
        pred = int(resp["Body"].read().decode("utf-8").strip().split("\n")[0])
        if i < 3:
            print(f"  request: {payload}  ->  pred: {pred}")
    print(f"\nInvoked {n_invocations} times. Capture files land in S3 within ~1 min.")
else:
    print("Skipped.")

  request: 0.7646806166083437,-0.6877444656872693,2.1055754397488986,0.565310270695689,-3.0523459235824113,-0.006910144925969752  ->  pred: 0
  request: -0.7310970194376203,0.05960098897463615,-2.4413113773095088,-1.7774123355490055,2.006076983849148,-0.7619967520535789  ->  pred: 1
  request: 0.1026709401606167,0.2727966245317937,0.38873244474812274,-2.054132538858891,-2.0856937493207877,-0.7252878604419573  ->  pred: 1

Invoked 50 times. Capture files land in S3 within ~1 min.


In [50]:
# Block 25b - Send a GENUINELY DRIFTED batch
# FIX vs v3: v3 only invoked with samples drawn from df[columns] — the same distribution the
# KS-test compares against — so drift was always ~0, the alarm never breached, and the
# auto-retrain loop never actually fired. This batch is shifted well outside the training
# distribution, so Block 26 reports real drift and the alarm in Block 29 genuinely trips.
SEND_DRIFTED_BATCH = True

if ALLOW_AWS_MUTATIONS and SEND_DRIFTED_BATCH:
    drifted = df[columns].sample(50, random_state=7, replace=True).copy()
    drifted["income_score"]      = drifted["income_score"] + 2.5
    drifted["debt_ratio_score"]  = drifted["debt_ratio_score"] - 2.0
    drifted["savings_score"]     = drifted["savings_score"] * 1.8
    for i, (_, row) in enumerate(drifted.iterrows()):
        payload = ",".join(str(v) for v in row.values)
        resp = runtime.invoke_endpoint(
            EndpointName=endpoint_name, ContentType="text/csv", Body=payload
        )
        _ = resp["Body"].read()
        if i < 3:
            print("  drifted request:", payload)
    print("\nSent 50 drifted requests. Block 26 should now report >= 3 drifted features,")
    print("which breaches DRIFT_FEATURE_COUNT_TO_RETRAIN and trips the alarm in Block 29.")
elif not SEND_DRIFTED_BATCH:
    print("SEND_DRIFTED_BATCH is False — drift will read ~0 and the alarm will not fire.")
else:
    print("Skipped.")

  drifted request: 1.5156350010167179,0.33177170510909826,-3.35360397892997,2.393455463771364,0.4852016570728999,1.139946271388829
  drifted request: 1.6848976608653328,-0.6508140072420673,-2.5412698242316933,-0.8434567928583916,-3.3171378699889327,0.33081616729648444
  drifted request: 3.681525803366013,0.3990716885796926,-1.728780494406867,-0.6305489942467611,-2.2004190219342132,-1.9656687313223338

Sent 50 drifted requests. Block 26 should now report >= 3 drifted features,
which breaches DRIFT_FEATURE_COUNT_TO_RETRAIN and trips the alarm in Block 29.


## 10. Real Drift Detection on Captured Requests

In [51]:
# Block 26 - Load captured requests from S3, KS-test vs training baseline
def load_captured_features(bucket, prefix_):
    paginator = s3.get_paginator("list_objects_v2")
    rows = []
    for page in paginator.paginate(Bucket=bucket, Prefix=prefix_):
        for obj in page.get("Contents", []):
            if not obj["Key"].endswith(".jsonl"):
                continue
            body = s3.get_object(Bucket=bucket, Key=obj["Key"])["Body"].read().decode("utf-8")
            for line in body.strip().split("\n"):
                if not line:
                    continue
                record = json.loads(line)
                inp = record.get("captureData", {}).get("endpointInput", {})
                data = inp.get("data", "")
                if inp.get("encoding") == "BASE64":
                    data = base64.b64decode(data).decode("utf-8")
                for csv_line in data.strip().split("\n"):
                    values = csv_line.split(",")
                    if len(values) == len(columns):
                        try:
                            rows.append([float(v) for v in values])
                        except ValueError:
                            continue
    return pd.DataFrame(rows, columns=columns) if rows else pd.DataFrame(columns=columns)

drifted_feature_count = 0
drift_report = None

if ALLOW_AWS_MUTATIONS:
    capture_prefix = f"{prefix}/data-capture"
    print("Waiting up to 3 min for capture files to appear...")
    captured = pd.DataFrame()
    for _ in range(18):
        captured = load_captured_features(bucket_name, capture_prefix)
        if len(captured) > 0:
            break
        time.sleep(10)
    if len(captured) == 0:
        print("No captured data yet — re-run in a minute.")
    else:
        print(f"Parsed {len(captured)} captured requests")
        baseline = df[columns]
        rows = []
        for col in columns:
            stat, p = ks_2samp(baseline[col], captured[col])
            rows.append({"feature": col, "ks_statistic": stat,
                         "p_value": p, "drifted": stat > DRIFT_KS_THRESHOLD})
        drift_report = pd.DataFrame(rows)
        drifted_feature_count = int(drift_report["drifted"].sum())
        print(drift_report)
        print("\nDrifted feature count:", drifted_feature_count)
else:
    print("Skipped.")

Waiting up to 3 min for capture files to appear...
Parsed 300 captured requests
                    feature  ks_statistic       p_value  drifted
0              income_score      0.346000  4.284011e-27     True
1      credit_history_score      0.090667  3.142966e-02    False
2          debt_ratio_score      0.352000  4.740473e-28     True
3          employment_score      0.067333  2.009916e-01    False
4             savings_score      0.168000  1.294209e-06     True
5  repayment_behavior_score      0.081333  7.053232e-02    False

Drifted feature count: 3


In [52]:
# Block 27 - Publish drift metric to CloudWatch
if ALLOW_AWS_MUTATIONS and drift_report is not None:
    cloudwatch.put_metric_data(
        Namespace=namespace,
        MetricData=[{
            "MetricName": "DriftedFeatureCount",
            "Value": float(drifted_feature_count),
            "Unit": "Count",
            "Dimensions": [{"Name": "Project", "Value": PROJECT_NAME}],
        }],
    )
    print("Published DriftedFeatureCount =", drifted_feature_count, "to", namespace)
else:
    print("Skipped.")

Published DriftedFeatureCount = 3 to unified-mlops-mlflow/dev


## 11. Wire the Alarm → SNS → Lambda → StartPipelineExecution

In [53]:
# Block 28 - Lambda function
lambda_code = f'''
import json
import boto3
def handler(event, context):
    client = boto3.client("sagemaker", region_name="{region}")
    resp = client.start_pipeline_execution(
        PipelineName="{pipeline_name}",
        PipelineExecutionDescription="Auto-triggered by drift alarm via SNS",
    )
    print("Started:", resp["PipelineExecutionArn"])
    return {{"statusCode": 200, "body": json.dumps({{"arn": resp["PipelineExecutionArn"]}})}}
'''

lambda_arn = None
if ALLOW_AWS_MUTATIONS:
    lambda_zip = local_dir / "retrain_lambda.zip"
    with zipfile.ZipFile(lambda_zip, "w") as zf:
        zf.writestr("lambda_function.py", lambda_code)
    with open(lambda_zip, "rb") as f:
        zip_bytes = f.read()

    try:
        resp = lambda_client.create_function(
            FunctionName=lambda_function_name,
            Runtime="python3.12",
            Role=execution_role,
            Handler="lambda_function.handler",
            Code={"ZipFile": zip_bytes},
            Timeout=60,
        )
        lambda_arn = resp["FunctionArn"]
        print("Lambda created:", lambda_arn)
    except lambda_client.exceptions.ResourceConflictException:
        lambda_client.update_function_code(FunctionName=lambda_function_name, ZipFile=zip_bytes)
        lambda_arn = lambda_client.get_function(FunctionName=lambda_function_name)["Configuration"]["FunctionArn"]
        print("Lambda updated:", lambda_arn)
else:
    print("Skipped.")

Lambda updated: arn:aws:lambda:eu-north-1:797715838180:function:unified-mlops-mlflow-dev-retrain-trigger


In [55]:
# Attach AmazonSNSFullAccess to the current role so Block 29's sns.create_topic works
_caller = sts.get_caller_identity()["Arn"]
if ":assumed-role/" in _caller:
    _role_name = _caller.split("/")[1]
else:
    _role_name = _caller.rsplit("/", 1)[-1]

_policy_arn = "arn:aws:iam::aws:policy/AmazonSNSFullAccess"

try:
    iam.attach_role_policy(RoleName=_role_name, PolicyArn=_policy_arn)
    print(f"OK - attached {_policy_arn}")
    print(f"     to role: {_role_name}")
except iam.exceptions.NoSuchEntityException as e:
    print(f"FAIL - {e}")
except Exception as e:
    # attach_role_policy is idempotent - re-attaching the same policy is a no-op success
    if "already attached" in str(e).lower():
        print(f"Already attached - proceeding.")
    else:
        raise

# Wait for IAM propagation
import time
print("\nWaiting 60s for IAM propagation...")
time.sleep(60)

# Verify it worked
try:
    sns.list_topics()
    print("OK - sns.list_topics() succeeded. Block 29 will now work.")
except Exception as e:
    print(f"Still denied: {e}")
    print("Wait another 30s and re-run this cell's verify step, or attach via Console.")

╭─────────────────────────────── Traceback (most recent call last) ────────────────────────────────╮
│ in <module>:11                                                                                   │
│                                                                                                  │
│    8 _policy_arn = "arn:aws:iam::aws:policy/AmazonSNSFullAccess"                                 │
│    9                                                                                             │
│   10 try:                                                                                        │
│ ❱ 11 │   iam.attach_role_policy(RoleName=_role_name, PolicyArn=_policy_arn)                      │
│   12 │   print(f"OK - attached {_policy_arn}")                                                   │
│   13 │   print(f"     to role: {_role_name}")                                                    │
│   14 except iam.exceptions.NoSuchEntityException as e:                                           │
│                                                                                                  │
│ /opt/conda/lib/python3.12/site-packages/botocore/client.py:606 in _api_call                      │
│                                                                                                  │
│    603 │   │   │   │   │   f"{py_operation_name}() only accepts keyword arguments."              │
│    604 │   │   │   │   )                                                                         │
│    605 │   │   │   # The "self" in this scope is referring to the BaseClient.                    │
│ ❱  606 │   │   │   return self._make_api_call(operation_name, kwargs)                            │
│    607 │   │                                                                                     │
│    608 │   │   _api_call.__name__ = str(py_operation_name)                                       │
│    609                                                                                           │
│                                                                                                  │
│ /opt/conda/lib/python3.12/site-packages/botocore/context.py:123 in wrapper                       │
│                                                                                                  │
│   120 │   │   │   with start_as_current_context():                                               │
│   121 │   │   │   │   if hook:                                                                   │
│   122 │   │   │   │   │   hook()                                                                 │
│ ❱ 123 │   │   │   │   return func(*args, **kwargs)                                               │
│   124 │   │                                                                                      │
│   125 │   │   return wrapper                                                                     │
│   126                                                                                            │
│                                                                                                  │
│ /opt/conda/lib/python3.12/site-packages/botocore/client.py:1094 in _make_api_call                │
│                                                                                                  │
│   1091 │   │   │   │   'error_code_override'                                                     │
│   1092 │   │   │   ) or error_info.get("Code")                                                   │
│   1093 │   │   │   error_class = self.exceptions.from_code(error_code)                           │
│ ❱ 1094 │   │   │   raise error_class(parsed_response, operation_name)                            │
│   1095 │   │   else:                                                                             │
│   1096 │   │   │   return parsed_response                                                        │
│   1097                                                     

In [57]:
# Block 29 - SNS topic + subscription + alarm wiring
if ALLOW_AWS_MUTATIONS and lambda_arn is not None:
    topic_arn = sns.create_topic(Name=sns_topic_name)["TopicArn"]
    print("SNS topic:", topic_arn)

    try:
        lambda_client.add_permission(
            FunctionName=lambda_function_name,
            StatementId="allow-sns-invoke",
            Action="lambda:InvokeFunction",
            Principal="sns.amazonaws.com",
            SourceArn=topic_arn,
        )
        print("Lambda permission added for SNS")
    except lambda_client.exceptions.ResourceConflictException:
        print("Lambda permission for SNS already present")

    existing = sns.list_subscriptions_by_topic(TopicArn=topic_arn).get("Subscriptions", [])
    if not any(s.get("Endpoint") == lambda_arn for s in existing):
        sns.subscribe(TopicArn=topic_arn, Protocol="lambda", Endpoint=lambda_arn)
        print("Lambda subscribed to SNS topic")
    else:
        print("Lambda already subscribed")

    cloudwatch.put_metric_alarm(
        AlarmName=alarm_name,
        Namespace=namespace,
        MetricName="DriftedFeatureCount",
        Dimensions=[{"Name": "Project", "Value": PROJECT_NAME}],
        Statistic="Maximum",
        Period=60,
        EvaluationPeriods=1,
        Threshold=DRIFT_FEATURE_COUNT_TO_RETRAIN,
        ComparisonOperator="GreaterThanOrEqualToThreshold",
        AlarmActions=[topic_arn],
        TreatMissingData="notBreaching",
    )
    print("Alarm wired: alarm → SNS → Lambda → StartPipelineExecution")
else:
    print("Skipped.")

## 12. Monitoring Summary

In [56]:
# Block 30 - Executions, registry, endpoint, alarm, S3
if ALLOW_AWS_MUTATIONS:
    print("=== Pipeline executions ===")
    try:
        for ex in sm.list_pipeline_executions(PipelineName=pipeline_name).get(
            "PipelineExecutionSummaries", []
        )[:5]:
            print(" ", ex["PipelineExecutionArn"], ex["PipelineExecutionStatus"])
    except ClientError:
        print("  (pipeline not upserted)")

    print("\n=== Model registry ===")
    try:
        for pkg in sm.list_model_packages(ModelPackageGroupName=model_package_group_name).get(
            "ModelPackageSummaryList", []
        )[:5]:
            print(" ", pkg["ModelPackageArn"], pkg["ModelApprovalStatus"])
    except ClientError:
        print("  (no group yet)")

    print("\n=== Endpoint ===")
    try:
        print(" ", sm.describe_endpoint(EndpointName=endpoint_name)["EndpointStatus"])
    except ClientError:
        print("  (no endpoint)")

    print("\n=== Alarm ===")
    try:
        for a in cloudwatch.describe_alarms(AlarmNames=[alarm_name])["MetricAlarms"]:
            print(" ", a["AlarmName"], "->", a["StateValue"])
    except ClientError:
        print("  (no alarm)")

    print("\n=== S3 artifacts ===")
    total = 0
    for page in s3.get_paginator("list_objects_v2").paginate(Bucket=bucket_name, Prefix=prefix):
        total += len(page.get("Contents", []))
    print(f"  Objects under {prefix}: {total}")

=== Pipeline executions ===
  arn:aws:sagemaker:eu-north-1:797715838180:pipeline/unified-mlops-mlflow-dev-training/execution/u49mt5b6ns6e Succeeded
  arn:aws:sagemaker:eu-north-1:797715838180:pipeline/unified-mlops-mlflow-dev-training/execution/241xxsovxbiq Succeeded
  arn:aws:sagemaker:eu-north-1:797715838180:pipeline/unified-mlops-mlflow-dev-training/execution/kfdvbthadvla Succeeded
  arn:aws:sagemaker:eu-north-1:797715838180:pipeline/unified-mlops-mlflow-dev-training/execution/a43t28j605m3 Succeeded
  arn:aws:sagemaker:eu-north-1:797715838180:pipeline/unified-mlops-mlflow-dev-training/execution/rt5scnv8fq00 Succeeded

=== Model registry ===
  arn:aws:sagemaker:eu-north-1:797715838180:model-package/unified-mlops-mlflow-dev-models/11 Approved
  arn:aws:sagemaker:eu-north-1:797715838180:model-package/unified-mlops-mlflow-dev-models/10 Approved
  arn:aws:sagemaker:eu-north-1:797715838180:model-package/unified-mlops-mlflow-dev-models/9 Approved
  arn:aws:sagemaker:eu-north-1:797715838180

## 13. Cleanup  *(guarded — set CLEANUP = True when you are done)*

In [ ]:
# Block 31 - Robust ordered teardown
CLEANUP = False

def safe(label, fn):
    try:
        fn()
        print("Deleted:", label)
    except ClientError as e:
        code_ = e.response.get("Error", {}).get("Code", "")
        msg = e.response.get("Error", {}).get("Message", "")
        print(f"Skip {label}: {code_} {msg}")
    except Exception as e:
        print(f"Skip {label}: {e}")

def wait_endpoint_gone():
    for _ in range(30):
        try:
            sm.describe_endpoint(EndpointName=endpoint_name)
            time.sleep(10)
        except ClientError:
            return
    print("  (endpoint still visible after 5 min — proceeding anyway)")

def wait_mlflow_gone():
    for _ in range(30):
        try:
            desc = sm.describe_mlflow_tracking_server(TrackingServerName=mlflow_server_name)
            if desc["TrackingServerStatus"] in ("Deleting", "Deleted"):
                pass
            time.sleep(10)
        except ClientError:
            return
    print("  (MLflow server still visible after 5 min — proceeding anyway)")

if CLEANUP and ALLOW_AWS_MUTATIONS:
    # 1. Endpoint first (uses the config + model)
    safe("endpoint", lambda: sm.delete_endpoint(EndpointName=endpoint_name))
    wait_endpoint_gone()

    # 2. All endpoint configs matching this endpoint's naming pattern
    try:
        for cfg in sm.list_endpoint_configs(NameContains=endpoint_name).get(
            "EndpointConfigs", []
        ):
            safe(f"endpoint-config {cfg['EndpointConfigName']}",
                 lambda n=cfg["EndpointConfigName"]: sm.delete_endpoint_config(EndpointConfigName=n))
    except ClientError as e:
        print("Skip endpoint-config listing:", e)

    # 3. All models matching this project's naming pattern
    try:
        for m in sm.list_models(NameContains=f"{PROJECT_NAME}-{ENVIRONMENT}-model").get("Models", []):
            safe(f"model {m['ModelName']}",
                 lambda n=m["ModelName"]: sm.delete_model(ModelName=n))
    except ClientError as e:
        print("Skip model listing:", e)

    # 4. Model packages (must come before deleting the group)
    try:
        for pkg in sm.list_model_packages(
            ModelPackageGroupName=model_package_group_name
        ).get("ModelPackageSummaryList", []):
            safe(f"model-package {pkg['ModelPackageArn'].split('/')[-1]}",
                 lambda a=pkg["ModelPackageArn"]: sm.delete_model_package(ModelPackageName=a))
    except ClientError as e:
        print("Skip model-package listing:", e)

    # 5. Model package group
    safe("model-package-group",
         lambda: sm.delete_model_package_group(ModelPackageGroupName=model_package_group_name))

    # 6. Pipeline
    safe("pipeline", lambda: sm.delete_pipeline(PipelineName=pipeline_name))

    # 7. Alarm
    safe("cloudwatch-alarm",
         lambda: cloudwatch.delete_alarms(AlarmNames=[alarm_name]))

    # 8. SNS subscriptions + topic
    try:
        topic_arn_cleanup = sns.create_topic(Name=sns_topic_name)["TopicArn"]  # idempotent, just to get the ARN
        for sub in sns.list_subscriptions_by_topic(TopicArn=topic_arn_cleanup).get("Subscriptions", []):
            arn = sub.get("SubscriptionArn", "")
            if arn and arn != "PendingConfirmation":
                safe(f"sns-subscription {arn.split(':')[-1]}",
                     lambda a=arn: sns.unsubscribe(SubscriptionArn=a))
        safe("sns-topic", lambda: sns.delete_topic(TopicArn=topic_arn_cleanup))
    except ClientError as e:
        print("Skip SNS:", e)

    # 9. Lambda
    safe("lambda", lambda: lambda_client.delete_function(FunctionName=lambda_function_name))

    # 10. CodeBuild project
    safe("codebuild-project",
         lambda: codebuild.delete_project(name=codebuild_project_name))

    # 11. ECR repo (force=True removes images too)
    safe("ecr-repository",
         lambda: ecr.delete_repository(repositoryName=ecr_repo_name, force=True))

    # 12. MLflow server (long-running deletion)
    safe("mlflow-tracking-server",
         lambda: sm.delete_mlflow_tracking_server(TrackingServerName=mlflow_server_name))
    wait_mlflow_gone()

    # 13. S3 objects under this project's prefix (paginated, handles >1000)
    print("Deleting S3 objects under", prefix, "...")
    deleted = 0
    for page in s3.get_paginator("list_objects_v2").paginate(Bucket=bucket_name, Prefix=prefix):
        objs = page.get("Contents", [])
        if not objs:
            continue
        for i in range(0, len(objs), 1000):
            batch = objs[i:i+1000]
            s3.delete_objects(
                Bucket=bucket_name,
                Delete={"Objects": [{"Key": o["Key"]} for o in batch]},
            )
            deleted += len(batch)
    print(f"Deleted {deleted} S3 objects under {prefix}")
    print("\nBucket kept intentionally (may contain other data). Delete manually if unused.")
else:
    print("CLEANUP is False (or ALLOW_AWS_MUTATIONS is False) — nothing deleted.")

## 14. Final Checklist

Every arrow below is a real AWS API call against a real service. No mocks, no simulated responses, no stubbed handlers.

Two honest caveats to keep off your slides unless you've checked them:

- **Section 7's ECR image is not the endpoint's serving path** unless you set `USE_BYOC_ENDPOINT = True` in Block 21b. By default the endpoint serves from the Model Registry package via the AWS-managed SKLearn inference container. Both are real; only one is live at a time.
- **This notebook has not been run end to end against AWS.** It compiles clean and the architecture is correct, but first-pass IAM and quota debugging should be expected.

```
S3 (data upload)
    ↓
SageMaker Pipeline (upserted DAG)
    Process → Train (logs to Managed MLflow) → Evaluate → ConditionStep (accuracy + f1) → Register
    ↓
Model Registry (auto-approved)
    ↓
Create Model + Endpoint Config (with DataCaptureConfig) → Endpoint
    ↓
Invoke Endpoint (50x) → Data Capture writes JSONL to S3
    ↓
KS-test on captured requests vs training baseline
    ↓
CloudWatch metric (DriftedFeatureCount)
    ↓
CloudWatch alarm (breach at threshold)
    ↓
SNS topic
    ↓
Lambda subscription (with add_permission for sns.amazonaws.com)
    ↓
sagemaker:StartPipelineExecution — back to the top, closed loop
```

**Loop closes automatically** — no cell needs to be re-run manually after the first end-to-end pass.